# Inference -- any fundus image -> every stage up to PDR

Give it any fundus photo (or a folder of them). Module 1 grades it and segments its lesions,
then Module 2 cascades stage by stage up to stage 4 (PDR): generate the next stage -> re-segment
it with Module 1 -> re-grade it (consistency check) -> feed it forward. Optionally Module 3
captions each panel with the estimated chance of further progression within 2 years.

Unlike notebook 08 (Tianjin patients only, stopping at their real follow-up grade), this needs no
dataset or follow-up image. Wraps `generate_progression.py`.

**Needs:** Module 1 checkpoints (notebooks 02/03) and a Module 2 Generator -- notebook 06 now
copies it to `Thesis_Datasets/module2_runs/<run>/final-G.ckpt` on Drive. Module 3 (notebook 07)
is optional.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os, glob, json
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'mod-2-NonLongIntegration'  # <-- change to 'main' once this branch is merged
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache')
MODULE2_RUNS_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs')
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
os.makedirs(MODULE2_RUNS_DIR, exist_ok=True)

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}
REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"
%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch matplotlib scipy torchmetrics torch-fidelity

In [ ]:
CLASSIFIER_RUN = 'fgadr'

# Locate the Module 1 classifier + segmentation checkpoints (same discovery logic as
# notebooks 04-08). HE/SE are included automatically once they exist on Drive.
SEG_DIR   = os.path.join(DRIVE_DATA_DIR, 'module1_runs', 'segmentation')
CLS_SAVES = os.path.join(DRIVE_DATA_DIR, 'module1_runs', 'classification', CLASSIFIER_RUN, 'saves')

_pref  = ['final_weights.pt', 'best_validation_weights.pt']
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02 (or 09 Part A) first"

SEG_ARGS = ''
for _lesion in ['EX', 'MA', 'HE', 'SE']:
    _p = os.path.join(SEG_DIR, f'models_FGADR_NO_TATL_{_lesion.lower()}', 'model_2.pth.tar')
    if os.path.isfile(_p):
        SEG_ARGS += f' --seg-checkpoint {_lesion}="{_p}"'
    print(f'{_lesion} seg :', _p if os.path.isfile(_p) else 'not trained yet')
assert 'EX=' in SEG_ARGS and 'MA=' in SEG_ARGS, 'EX/MA segmentation checkpoints missing -- run notebook 03 first'
print('classifier:', CLS_CKPT)

# Module 2 Generator: first one found, preferring the EyePACS-pretrained + fine-tuned run.
_gen_cands = [
    os.path.join(MODULE2_RUNS_DIR, 'models_poc_tianjin_from_nonlong', 'final-G.ckpt'),
    os.path.join(MODULE2_RUNS_DIR, 'models_poc_tianjin', 'final-G.ckpt'),
    f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/models_poc_tianjin_from_nonlong/final-G.ckpt',
    f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/models_poc_tianjin/final-G.ckpt',
]
GENERATOR_CKPT = next((p for p in _gen_cands if os.path.isfile(p)), None)
assert GENERATOR_CKPT, 'No Module 2 final-G.ckpt found -- run notebook 06 first'
print('generator :', GENERATOR_CKPT)

# Module 3 (optional): notebook 07's outputs, on Drive (Thesis_Datasets/module3_runs_poc/) or in this runtime.
MODULE3_DIR = next((d for d in (os.path.join(DRIVE_DATA_DIR, 'module3_runs_poc'),
                                 f'{REPO_CODE_DIR}/module3/module3_runs_poc')
                    if os.path.isfile(os.path.join(d, 'final-model.ckpt'))), None)
MODULE3_ARGS = ''
if MODULE3_DIR:
    MODULE3_ARGS = (f' --module3-checkpoint "{MODULE3_DIR}/final-model.ckpt"'
                    f' --module3-thresholds-json "{MODULE3_DIR}/poc_results.json"')
print('module 3  :', MODULE3_ARGS or 'not found -- figures without progression-probability captions')

In [ ]:
# Input: a Drive folder of fundus images, or upload some now if that folder is empty/missing.
INPUT_DIR = os.path.join(DRIVE_DATA_DIR, 'progression_inputs')
os.makedirs(INPUT_DIR, exist_ok=True)
if not glob.glob(os.path.join(INPUT_DIR, '*')):
    from google.colab import files
    for _name, _data in files.upload().items():
        open(os.path.join(INPUT_DIR, _name), 'wb').write(_data)
print(sorted(os.listdir(INPUT_DIR)))

In [ ]:
START_STAGE = None   # None = use Module 1's grade of the input; or force a known clinical grade 0-3
END_STAGE = 4        # 4 = PDR
OUT_DIR = os.path.join(DRIVE_DATA_DIR, 'progression_outputs')

_start = f' --start-stage {START_STAGE}' if START_STAGE is not None else ''
%cd {REPO_CODE_DIR}
!python generate_progression.py --input "{INPUT_DIR}" \
    --generator-checkpoint "{GENERATOR_CKPT}" \
    --classifier-checkpoint "{CLS_CKPT}"{SEG_ARGS}{MODULE3_ARGS} \
    --end-stage {END_STAGE}{_start} --out-dir "{OUT_DIR}"

In [ ]:
from IPython.display import Image as IPImage, display
for _fig in sorted(glob.glob(f'{OUT_DIR}/*/trajectory.png')):
    _summary = json.load(open(_fig.replace('trajectory.png', 'trajectory.json')))
    print(f"\n{_summary['input']}: Module 1 grade {_summary['module1_input_grade']}, start stage {_summary['start_stage']}")
    for _s in _summary['steps']:
        print(f"  stage {_s['stage']} ({_s['stage_name']}): LBS {_s['lbs']:.4f}"
              + (f", Module 1 re-grade {_s['module1_grade']} ({'consistent' if _s['module1_consistent'] else 'NOT consistent'})" if _s['synthesized'] else ' [input]'))
    display(IPImage(_fig))

## Caveats

- Module 1 does not segment neovascularisation (NV), the defining sign of PDR, so stage-4
  images mostly show heavier EX/HE/MA/SE unless the Generator learned NV from EyePACS stage-4
  images during pretraining (notebook 09).
- Each step builds on the previous synthesized image, so errors compound -- the PDR panel is
  the least reliable.
- "Module 1 re-grade ... NOT consistent" can mean the synthesized image is implausible, or that
  Module 1 is unreliable on synthesized images, or both.
- Module 3 captions on synthesized panels are outside its training distribution (untested).